# Demo 03d — Classification Metrics

**Class 03 · Block 4** (slides: `slides/03_linear_models.md`)

One live demo on the **spam** data (Spambase: 4601 e-mails, 57 word/character frequencies, 39% spam):

* **C1 — Classification metrics:** confusion matrix, accuracy, precision, recall, $F_1$, MCC, and the threshold.

The classifier is the regularized logistic regression of `03_logistic_regression.ipynb`.

In [ ]:
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../datasets")

In [ ]:
spam = pl.read_csv(DATA / "spambase.csv.zst")
names = [c for c in spam.columns if c != "spam"]
X_raw = spam.select(names).to_numpy().astype(float)
y = spam["spam"].to_numpy().astype(int)
idx_tr, idx_te = (np.asarray(a) for a in train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=0))
X_tr_raw, X_te_raw, ytr, yte = X_raw[idx_tr], X_raw[idx_te], y[idx_tr], y[idx_te]
# Most frequencies are 0 and a few are huge: log(1 + x), then standardize with the training statistics.
scaler = StandardScaler().fit(np.log1p(X_tr_raw))
Xtr, Xte = np.asarray(scaler.transform(np.log1p(X_tr_raw))), np.asarray(scaler.transform(np.log1p(X_te_raw)))
print("train", Xtr.shape, "test", Xte.shape, "| spam fraction: train", ytr.mean().round(3), "test", yte.mean().round(3))
print("duplicate e-mails in the data:", int(spam.is_duplicated().sum()), "of", spam.height, "(a leak between train and test)")

### The classifier

A regularized logistic regression on the log-standardized features (`C = 0.1`); its probabilities are what the metrics judge.

In [ ]:
reg = LogisticRegression(C=0.1, max_iter=10_000).fit(Xtr, ytr)

## C1 — Classification metrics

The **confusion matrix** counts the four kinds of outcome (spam is the positive class):

| | predicted spam | predicted ham |
|:--|:-:|:-:|
| **actual spam** | TP | FN |
| **actual ham** | FP | TN |

$$\text{accuracy} = \tfrac{TP+TN}{n},\quad \text{precision} = \tfrac{TP}{TP+FP},\quad \text{recall} = \tfrac{TP}{TP+FN},\quad
F_1 = \tfrac{2PR}{P+R},\quad
\text{MCC} = \tfrac{TP\,TN - FP\,FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

In [ ]:
def metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "accuracy": (tp + tn) / len(y_true),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }


proba = reg.predict_proba(Xte)[:, 1]
pred = (proba >= 0.5).astype(int)
m = metrics(yte, pred)
print({k: (int(v) if k in ("TP", "FP", "FN", "TN") else round(float(v), 3)) for k, v in m.items()})
ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=["ham", "spam"], cmap="Blues")
plt.show()

**Accuracy hides imbalance.** Keep all the ham but only 2% spam in the test set (a mailbox with rare spam), and
compare the classifier with the lazy baseline "everything is ham".

In [ ]:
rng = np.random.default_rng(0)
ham, spam_idx = np.flatnonzero(yte == 0), np.flatnonzero(yte == 1)
keep = np.r_[ham, rng.choice(spam_idx, int(round(0.02 * len(ham) / 0.98)), replace=False)]
y_imb, p_imb = yte[keep], pred[keep]
rows = {
    "classifier": metrics(y_imb, p_imb),
    "everything is ham": {"accuracy": float(np.mean(y_imb == 0)), "precision": 0.0, "recall": 0.0, "F1": 0.0, "MCC": 0.0},
}
print(f"imbalanced test set: {len(y_imb)} e-mails, {int(y_imb.sum())} spam ({100 * y_imb.mean():.1f}%)")
for name, r in rows.items():
    print(f"  {name:<18}", {k: round(float(r[k]), 3) for k in ("accuracy", "precision", "recall", "F1", "MCC")})

Accuracy of the lazy baseline is about 98%: **the metric must match the goal**. And when a model calls everything
positive, recall is 1 and $F_1$ still looks reasonable; **MCC** (which uses all four cells) is 0. MCC is +1 for perfect
predictions, 0 for chance and $-1$ for total disagreement, whatever the class balance.

In [ ]:
all_spam = np.ones_like(yte)
print("everything is spam:", {k: round(float(v), 3) for k, v in metrics(yte, all_spam).items() if k in ("accuracy", "precision", "recall", "F1", "MCC")})

**The threshold is a decision.** The classifier returns a probability; 0.5 is just a default. A false positive
(hiding a real e-mail) costs more than a false negative (one more spam in the inbox), so we raise the threshold.

In [ ]:
ts = np.linspace(0.02, 0.98, 49)
rows = [metrics(yte, (proba >= t).astype(int)) for t in ts]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for key in ("precision", "recall", "F1", "MCC"):
    axes[0].plot(ts, [r[key] for r in rows], label=key)
axes[0].set(xlabel="threshold", ylim=(0, 1.02))
axes[0].legend(ncol=4, loc="lower center")
prec, rec, _ = precision_recall_curve(yte, proba)
axes[1].plot(rec, prec)
axes[1].set(xlabel="recall", ylabel="precision", title=f"precision-recall curve (AP {average_precision_score(yte, proba):.3f})")
plt.tight_layout()
plt.show()
print("ROC AUC (probability that a random spam scores above a random ham):", round(float(roc_auc_score(yte, proba)), 4))
for t in (0.5, 0.9, 0.99):
    r = metrics(yte, (proba >= t).astype(int))
    print(f"threshold {t}: precision {r['precision']:.3f}, recall {r['recall']:.3f}, F1 {r['F1']:.3f}, MCC {r['MCC']:.3f}, ham lost {r['FP']}")

## Summary

* The confusion matrix is the source of all the metrics; accuracy fails on imbalance, $F_1$ ignores the true negatives, **MCC** uses all four cells.
* The decision threshold is a separate, cost-driven choice; a precision-recall curve and the ROC AUC summarize all thresholds.